# A voice for the concierge

**Week 11 · Session 2 · Notebook 3 of 3**

Voice sounds like a new kind of AI. It isn't. A voice agent is **the same agent**, with
speech-to-text in front and text-to-speech behind:

```
 your voice ──► STT ──► text ──► [ the agent you already built ] ──► text ──► TTS ──► its voice
```

We'll build that three ways:

1. **By hand** — the raw speech APIs, so you see there's nothing hidden.
2. **With `VoicePipeline`** — the SDK's version, streaming audio back.
3. **With the multi-agent concierge** — handoffs work unchanged in voice.

Then we measure where the time goes, which is why *realtime* speech-to-speech models exist.

> **No microphone needed.** We use text-to-speech to *generate* the traveller's spoken
> question, so the notebook runs anywhere — Colab included. `voice_live.py` in this folder
> does the same thing from a real microphone.

In [1]:
%pip install -q "openai-agents[voice]==0.22.3" python-dotenv numpy

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.5.3 which is incompatible.
scipy 1.13.1 requires numpy<2.3,>=1.22.4, but you have numpy 2.5.3 which is incompatible.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.5.3 which is incompatible.
ydata-profiling 4.16.1 requires numpy<2.2,>=1.16.0, but you have numpy 2.5.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.3 which is incompatible.
liquid-audio 1.0.0 requires torch>=2.8.0, but you have torch 2.6.0 which is incompatible.
llama-index-llms-openai 0.5.6 requires openai<2,>=1.81.0, but you have openai 3.16.2 which is incompatible.
streamlit 1.37.1 requires pillow<11,>=7.1.0, but you have pillow 12.0.0 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.32.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, io, json, time, wave, getpass
from pathlib import Path
import numpy as np

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from openai import OpenAI
from agents import Agent, Runner, function_tool
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from IPython.display import Audio, display

# httpx2 2.12+ decompresses brotli with output_buffer_limit=... as a keyword.
# Anaconda's brotli 1.x exposes process() as a C function that rejects kwargs
# ("TypeError: process() takes no keyword arguments"). Wrap it so OpenAI
# Responses (which often arrive Content-Encoding: br) can be decoded.
try:
    import brotli as _brotli
    from httpx2._decoders import BrotliDecoder as _BrotliDecoder
    _probe = _brotli.Decompressor()
    try:
        _probe.process(b"", output_buffer_limit=1)
    except TypeError:
        _brotli_init = _BrotliDecoder.__init__
        def _brotli_init_compat(self, *args, **kwargs):
            _brotli_init(self, *args, **kwargs)
            _impl = self._decompress
            self._decompress = lambda data, output_buffer_limit=None, **_k: _impl(data)
        _BrotliDecoder.__init__ = _brotli_init_compat
except Exception:
    pass

client = OpenAI()
MODEL = "gpt-4.1-mini"
SAMPLE_RATE = 24_000          # OpenAI TTS returns 24 kHz, 16-bit, mono PCM
print("ready")

ready


In [3]:
def pcm_to_wav(pcm: np.ndarray, path: str, rate: int = SAMPLE_RATE) -> str:
    """Wrap raw 16-bit mono PCM in a WAV header so any player (and STT) can read it."""
    with wave.open(path, "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate)
        w.writeframes(pcm.astype(np.int16).tobytes())
    return path


def speak(text: str, path: str, voice: str = "alloy") -> np.ndarray:
    """Text -> speech. Streams raw PCM (avoids WAV-header quirks), saves a .wav."""
    with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts", voice=voice, input=text, response_format="pcm") as resp:
        pcm = np.frombuffer(b"".join(resp.iter_bytes()), dtype=np.int16)
    pcm_to_wav(pcm, path)
    return pcm

---
# Part 1 — Speech is just an API

## Text-to-speech: make the traveller's voice

In [4]:
QUESTION = ("Hi! I'm flying from Bangalore to Goa on the second of October "
            "twenty twenty-six. What's the cheapest flight?")

t0 = time.time()
question_pcm = speak(QUESTION, "question.wav", voice="alloy")
print(f"{len(question_pcm) / SAMPLE_RATE:.1f}s of audio in {time.time() - t0:.1f}s -> question.wav")
display(Audio("question.wav"))

6.9s of audio in 5.2s -> question.wav


## Speech-to-text: Whisper, and what replaced it

`whisper-1` is the model most tutorials still use. OpenAI's current transcription models
are `gpt-4o-transcribe` and `gpt-4o-mini-transcribe`: better on accents, noise and
domain words. Same API call, different model name.

In [5]:
for model in ["whisper-1", "gpt-4o-mini-transcribe", "gpt-4o-transcribe"]:
    t0 = time.time()
    with open("question.wav", "rb") as f:
        text = client.audio.transcriptions.create(model=model, file=f).text
    print(f"{model:<24} {time.time() - t0:>4.1f}s   {text}")

whisper-1                 1.9s   Hi, I'm flying from Bangalore to Goa on the 2nd of October 2026. What's the cheapest flight?


gpt-4o-mini-transcribe    1.3s   Hi, I'm flying from Bangalore to Goa on the 2nd of October 2026. What's the cheapest flight?


gpt-4o-transcribe         1.6s   Hi, I'm flying from Bangalore to Goa on the 2nd of October 2026. What's the cheapest flight?


All three probably got it right: this is clean, synthetic speech. The differences show up
on real microphones, in noisy rooms, and with names like *"Candolim"* or *"Akasa Air"*.
Record yourself saying the sentence (or run `voice_live.py`) and compare.

---
# Part 2 — A voice agent by hand

Here's a small version of yesterday's concierge. The one change for voice is in the
**instructions**: no markdown, no lists, no symbols, and short answers. A voice can't say
a bullet point, and nobody wants to listen to a paragraph.

In [6]:
FLIGHTS = [
    {"id": "6E-512",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "07:10", "airline": "IndiGo",    "price_inr": 4180},
    {"id": "AI-887",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "13:45", "airline": "Air India", "price_inr": 5620},
    {"id": "QP-1375", "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "18:30", "airline": "Akasa Air", "price_inr": 3890},
]
HOTELS = {"goa": [{"name": "Fontainhas Heritage Stay", "area": "Panjim",         "price_inr": 3800},
                  {"name": "Sea Breeze Candolim",      "area": "Candolim beach", "price_inr": 5200}]}


@function_tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search flights between two airports (IATA codes) on a date (YYYY-MM-DD), cheapest first."""
    hits = sorted((f for f in FLIGHTS if f["from"] == origin.upper() and f["to"] == destination.upper()
                   and f["date"] == date), key=lambda f: f["price_inr"])
    return json.dumps(hits) if hits else "No flights found."


@function_tool
def search_hotels(city: str, max_price_inr: int = 100000) -> str:
    """Find hotels in a city under a nightly budget, cheapest first."""
    opts = [h for h in HOTELS.get(city.lower(), []) if h["price_inr"] <= max_price_inr]
    return json.dumps(sorted(opts, key=lambda h: h["price_inr"])) if opts else "No hotels found."


VOICE_RULES = ("You are speaking out loud, on a phone call. Reply in at most two short sentences. "
               "No lists, no markdown, no symbols, no emoji. Say prices as words a person would say, "
               "like 'three thousand eight hundred and ninety rupees'. Say times like 'six thirty in the evening'.")

voice_concierge = Agent(
    name="Voice Concierge",
    instructions=f"You are a travel concierge. Always search before quoting. {VOICE_RULES}",
    model=MODEL,
    tools=[search_flights, search_hotels],
)
print("agent ready")

agent ready


In [7]:
timings = {}

# 1. ears
t = time.time()
with open("question.wav", "rb") as f:
    heard = client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=f).text
timings["speech-to-text"] = time.time() - t

# 2. brain -- the exact agent loop from yesterday
t = time.time()
answer = (await Runner.run(voice_concierge, heard)).final_output
timings["agent (model + tools)"] = time.time() - t

# 3. mouth
t = time.time()
speak(answer, "answer_by_hand.wav", voice="nova")
timings["text-to-speech"] = time.time() - t

print(f"HEARD : {heard}\nSAID  : {answer}\n")
for stage, secs in timings.items():
    print(f"  {stage:<24} {secs:>5.1f}s")
print(f"  {'total':<24} {sum(timings.values()):>5.1f}s")
display(Audio("answer_by_hand.wav"))

HEARD : Hi, I'm flying from Bangalore to Goa on the 2nd of October, 2026. What's the cheapest flight?
SAID  : The cheapest flight from Bangalore to Goa on the 2nd of October is with Akasa Air, departing at six thirty in the evening, priced at three thousand eight hundred and ninety rupees.

  speech-to-text             1.5s
  agent (model + tools)      2.7s
  text-to-speech             2.7s
  total                      6.9s


That's a working voice agent in three calls. The agent in the middle is *unchanged*:
same tools, same loop, same `Runner.run`.

Look at the total, though. Several seconds of silence on a phone call feels broken.
Two problems: every stage waits for the previous one to **finish**, and we only start
speaking after the **whole** answer is written.

---
# Part 3 — `VoicePipeline`: the same thing, streamed

The SDK's `VoicePipeline` wraps exactly those three stages, but **streams** them: TTS
starts speaking the first sentence while the agent is still writing the second.

In [8]:
from agents.voice import (VoicePipeline, SingleAgentVoiceWorkflow, AudioInput,
                          VoicePipelineConfig, TTSModelSettings)

pipeline = VoicePipeline(
    workflow=SingleAgentVoiceWorkflow(voice_concierge),
    config=VoicePipelineConfig(
        tts_settings=TTSModelSettings(voice="nova"),
        tracing_disabled=True,     # voice traces carry raw audio; keep them off in class
    ),
)

audio_in = AudioInput(buffer=question_pcm, frame_rate=SAMPLE_RATE)

t0 = time.time()
first_audio_at = None
chunks = []
result = await pipeline.run(audio_in)
async for event in result.stream():
    if event.type == "voice_stream_event_audio":
        if first_audio_at is None:
            first_audio_at = time.time() - t0      # when the traveller would start hearing it
        chunks.append(event.data)

reply = np.concatenate(chunks)
pcm_to_wav(reply, "answer_pipeline.wav")
print(f"first audio after {first_audio_at:.1f}s  ·  whole reply ready after {time.time() - t0:.1f}s"
      f"  ·  {len(reply) / SAMPLE_RATE:.1f}s of speech")
display(Audio("answer_pipeline.wav"))

first audio after 6.8s  ·  whole reply ready after 7.7s  ·  11.6s of speech


The number that matters is **time to first audio**: how long the traveller waits before
they hear *anything*. Compare it with the by-hand total above.

You'll probably find **streaming barely helped.** That's the honest lesson here. Streaming
only speeds up the *last* stage, speaking. Most of the wait happens before the first word
of the answer even exists: transcribing the whole question, then a model call, a tool
call, and another model call. On a two-sentence answer there's almost nothing left for
streaming to save. It pays off on long answers — and Part 5 shows the fix for short ones.

`pipeline.run` hands back an audio **stream**. In a real app you'd push each chunk to a
speaker or a phone line as it arrives. Here we collect the chunks and save a file.

The SDK's defaults are the current models: `gpt-4o-transcribe` for ears and
`gpt-4o-mini-tts` for the mouth. Swap them with `VoicePipeline(stt_model=..., tts_model=...)`.

---
# Part 4 — Voice + multi-agent

Everything from `02_multi_agent_guardrails` works in voice, because voice is only I/O.
Here's a front desk that hands off to a flights desk or a hotels desk: same handoffs,
now spoken.

In [9]:
flights_desk = Agent(
    name="Flights Desk", handoff_description="Flight search.", model=MODEL,
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the flights desk. Always search first. {VOICE_RULES}",
    tools=[search_flights],
)
hotels_desk = Agent(
    name="Hotels Desk", handoff_description="Hotel search.", model=MODEL,
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nYou are the hotels desk. Always search first. {VOICE_RULES}",
    tools=[search_hotels],
)
front_desk = Agent(
    name="Front Desk", model=MODEL,
    instructions=f"{RECOMMENDED_PROMPT_PREFIX}\nRoute the caller to the right desk. {VOICE_RULES}",
    handoffs=[flights_desk, hotels_desk],
)


class TrackingWorkflow(SingleAgentVoiceWorkflow):
    """Same workflow, but remembers which agent answered -- so we can print it."""
    def __init__(self, agent):
        super().__init__(agent)
        self.answered_by = []

    async def run(self, transcription):
        async for text in super().run(transcription):
            yield text
        # _current_agent is internal: the workflow sets it to result.last_agent after each turn.
        # Fine for a demo; in production, subclass VoiceWorkflowBase and track it yourself.
        self.answered_by.append(self._current_agent.name)


workflow = TrackingWorkflow(front_desk)
desk_pipeline = VoicePipeline(workflow=workflow, config=VoicePipelineConfig(
    tts_settings=TTSModelSettings(voice="shimmer"), tracing_disabled=True))

caller_pcm = speak("Hello, I need a hotel in Goa for under five thousand rupees a night.", "caller.wav")
display(Audio("caller.wav"))

result = await desk_pipeline.run(AudioInput(buffer=caller_pcm, frame_rate=SAMPLE_RATE))
chunks = [e.data async for e in result.stream() if e.type == "voice_stream_event_audio"]
pcm_to_wav(np.concatenate(chunks), "desk_answer.wav")
print("answered by:", workflow.answered_by[-1])
display(Audio("desk_answer.wav"))

answered by: Hotels Desk


The caller spoke to the Front Desk, the Front Desk handed off, and the **Hotels Desk
answered in voice**. Nothing about handoffs changed. Guardrails and approvals carry over
the same way, though an approval in voice means *asking out loud* — a good exercise.

---
# Part 5 — Where the time goes, and why realtime exists

Our pipeline is **chained**: speech → text → agent → text → speech. Each arrow is a
separate model, and the hand-offs between them cost time. Our measured budget:

In [10]:
budget = dict(timings)
print(f"{'stage':<26}{'seconds':>8}")
print("-" * 34)
for stage, secs in budget.items():
    bar = "█" * int(secs * 6)
    print(f"{stage:<26}{secs:>8.1f}  {bar}")
print("-" * 34)
print(f"{'by hand, total':<26}{sum(budget.values()):>8.1f}")
print(f"{'pipeline, first audio':<26}{first_audio_at:>8.1f}")
print("\nHumans notice a pause after roughly half a second.")

stage                      seconds
----------------------------------
speech-to-text                 1.5  █████████
agent (model + tools)          2.7  ████████████████
text-to-speech                 2.7  ████████████████
----------------------------------
by hand, total                 6.9
pipeline, first audio          6.8

Humans notice a pause after roughly half a second.


Even streamed, a chained pipeline takes seconds before the first word. Real phone
conversations need well under a second. That's what **speech-to-speech** models are
for: one model hears audio and speaks audio directly — `gpt-realtime` — and the SDK
wraps it as a `RealtimeAgent`:

```python
from agents.realtime import RealtimeAgent, RealtimeRunner

agent  = RealtimeAgent(name="Concierge", instructions=VOICE_RULES, tools=[search_flights])
runner = RealtimeRunner(agent)          # streams mic audio in and speaker audio out over a websocket
```

It needs a live audio stream in both directions, so it belongs in a browser or phone
app rather than a notebook. We won't build one today, but the agent inside it is the one
you already know.

| | Chained (`VoicePipeline`) | Speech-to-speech (`RealtimeAgent`) |
|---|---|---|
| Latency | seconds | sub-second |
| Can see the text in between | yes — log it, check it, guard it | harder |
| Pick any text model | yes | no — realtime models only |
| Cost | lower | higher |
| Good for | voice notes, IVR, assistants that can wait | live phone calls, conversation |

**The honest trade-off:** chained is slower but gives you a text transcript at every step,
so your guardrails, logging and evals all keep working. Speech-to-speech is fast but
harder to inspect. Pick by how much latency your users will tolerate.

---
## Exercises

**1. Spoken approval.** Give the voice concierge `book_flight(needs_approval=True)`. When
the run pauses, *speak* the confirmation question ("Shall I book QP-1375 for three
thousand eight hundred and ninety rupees?"), transcribe the caller's reply, and
approve or reject.

**2. Break the voice rules.** Remove `VOICE_RULES` and listen to the result. How does it
read out a markdown list?

**3. Accents.** Record the question yourself with `voice_live.py`, then compare
`whisper-1` and `gpt-4o-transcribe` on it. Try a noisy room.

**4. A Hindi desk.** Add a desk that replies in Hindi (the TTS voices can speak it) and
route to it when the caller speaks Hindi.